### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 2: Simple mathematical intuitions. Exercise 3**

**Gradient descent**

We go back to the fit of chapter 1: gene expression $y$ measured at different doses of a stimulus $x$. The **input** $x$ goes into a **model** (the line), which gives a prediction $y^{\text{pred}}$, and we compare it with the **label** $y$ using a **loss**.
In chapter 1 we used the best fit formula. Now we start from one line and let the **derivatives** of the loss tell us in which direction to move the slope and the intercept, step by step, until we reach the same best line.

**The maths**

For every entry $i$ we have a label $y_i$, the true value, and a prediction $y_i^{\text{pred}}$, the output of the model. The loss is the sum of the squared differences between them:

$$L = \sum_{i} \left( y_i - y_i^{\text{pred}} \right)^2$$

The smaller the loss, the closer the predictions are to the labels.


Here $i$ runs over the 11 measurements. The prediction is a straight line with slope $a$ and intercept $b$:

$$y_i^{\text{pred}} = a\,x_i + b$$

The loss depends on two numbers, $a$ and $b$, so it has two derivatives: one when we change only $a$, and one when we change only $b$ (partial derivatives):

$$\frac{\partial L}{\partial a} = -2 \sum_{i} x_i \left( y_i - y_i^{\text{pred}} \right) \qquad \frac{\partial L}{\partial b} = -2 \sum_{i} \left( y_i - y_i^{\text{pred}} \right)$$

Gradient descent moves both a small step against their derivative, so the loss goes down. The learning rate $\eta$ sets the size of the step:

$$a \leftarrow a - \eta \, \frac{\partial L}{\partial a} \qquad b \leftarrow b - \eta \, \frac{\partial L}{\partial b}$$

**Where the best fit formula comes from.** At the minimum both derivatives are zero. Setting $\partial L / \partial b = 0$ and then $\partial L / \partial a = 0$, and writing $\bar{x}$ and $\bar{y}$ for the means, gives exactly the formula we used in chapter 1:

$$b = \bar{y} - a\,\bar{x} \qquad a = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}$$

The formula is the minimum found in one go with the derivatives. Gradient descent reaches the same point step by step.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

# Check installation
print("All libraries imported successfully")
print(np.__version__)

In [ ]:
# Input: doses of the stimulus
x = np.linspace(0, 10, 11)

# Label: measured gene expression (a hidden straight line plus some noise)
np.random.seed(0)
y = 0.5 * x + 1.2 + np.random.normal(0, 0.4, size = x.shape)

print("x:", x)
print("y:", np.round(y, 2))
print("Shape:", x.shape)
print("Format: ", type(x))

In [ ]:
# Plot the measurements
plt.figure(figsize = (8, 5))
plt.scatter(x, y, color = 'blue', label = 'Measurements (label y)')
plt.xlabel('x: stimulus (input)')
plt.ylabel('y: gene expression')
plt.title('Experimental data')
plt.legend()
plt.show()

In [ ]:
# Model: a straight line. Input x -> output y_pred
def model(x, a, b):
    return a * x + b

In [ ]:
# Loss: sum of squared differences between the label y and the prediction y_pred
def loss(y, y_pred):
    return np.sum((y - y_pred)**2)

In [ ]:
# Table: input, label, output and squared error for each measurement
def print_table(x, y, y_pred):
    print(" input x | label y | output y_pred | (y - y_pred)^2")
    print("-" * 50)
    for xi, yi, yhi in zip(x, y, y_pred):
        print(f"  {xi:5.1f}  |  {yi:5.2f}  |    {yhi:5.2f}     |     {(yi - yhi)**2:5.2f}")
    print("-" * 50)
    print(f"Loss (sum of the last column): {loss(y, y_pred):.3f}")

# Plot: the line with its errors (left) and the squared error of each point (right)
def plot_loss(x, y, y_pred, title):
    fig, axes = plt.subplots(1, 2, figsize = (14, 5))

    axes[0].scatter(x, y, color = 'blue', label = 'Label y')
    axes[0].plot(x, y_pred, color = 'red', marker = 'o', label = 'Output y_pred')
    for xi, yi, yhi in zip(x, y, y_pred):
        axes[0].vlines(xi, yhi, yi, color = 'gray', linestyle = 'dashed')
    axes[0].set_xlabel('x: stimulus (input)')
    axes[0].set_ylabel('y: gene expression')
    axes[0].set_title(title)
    axes[0].legend()

    axes[1].bar(x, (y - y_pred)**2, color = 'gray')
    axes[1].set_xlabel('x: stimulus (input)')
    axes[1].set_ylabel('(y - y_pred)^2')
    axes[1].set_title(f'Error of each measurement. Loss = sum of the bars = {loss(y, y_pred):.3f}')

    plt.tight_layout()
    plt.show()

In [ ]:
# Derivatives of the loss with respect to the slope a and the intercept b
def loss_derivatives(a, b):
    error = y - model(x, a, b)
    dL_da = -2 * np.sum(x * error)
    dL_db = -2 * np.sum(error)
    return dL_da, dL_db

In [ ]:
# Starting line: slope 0, intercept 0
a, b = 0.0, 0.0
y_pred = model(x, a, b)
print_table(x, y, y_pred)
dL_da, dL_db = loss_derivatives(a, b)
print(f"dL/da = {dL_da:.2f}, dL/db = {dL_db:.2f}  (negative: increase a and b to go down)")
plot_loss(x, y, y_pred, f'Start: y_pred = {a} x + {b}')

In [ ]:
# Gradient descent implementation
# Try learning_rate = 0.0005 (slow) or 0.003 (the steps are too big and it jumps away)
learning_rate = 0.002
path_a, path_b, path_loss = [a], [b], [loss(y, model(x, a, b))]

print("Gradient descent running")
for step in range(1000):
    dL_da, dL_db = loss_derivatives(a, b)
    a = a - learning_rate * dL_da
    b = b - learning_rate * dL_db
    path_a.append(a)
    path_b.append(b)
    path_loss.append(loss(y, model(x, a, b)))
    if step % 100 == 0 or step == 999:
        print(f"Step {step}: a = {a:.4f}, b = {b:.4f}, loss = {path_loss[-1]:.4f}")

In [ ]:
# The line after some of the steps: it moves towards the measurements
plt.figure(figsize = (8, 5))
plt.scatter(x, y, color = 'blue', label = 'Label y')
for step in [0, 1, 10, 100, 1000]:
    plt.plot(x, model(x, path_a[step], path_b[step]), label = f'Step {step}: y_pred = {path_a[step]:.2f} x + {path_b[step]:.2f}')
plt.xlabel('x: stimulus (input)')
plt.ylabel('y: gene expression')
plt.title('The line during gradient descent')
plt.legend()
plt.show()

In [ ]:
# The loss goes down at every step
plt.figure(figsize = (8, 5))
plt.plot(path_loss, color = 'purple')
plt.yscale('log')
plt.xlabel('step')
plt.ylabel('loss L (log scale)')
plt.title('Loss during gradient descent')
plt.grid(True)
plt.show()

In [ ]:
# Map of the loss for every slope a and intercept b, like a topographic map
# Darker means lower loss. The red path is gradient descent walking downhill
a_grid, b_grid = np.meshgrid(np.linspace(-0.2, 1.2, 200), np.linspace(-0.5, 3.0, 200))
loss_grid = np.zeros_like(a_grid)
for i in range(a_grid.shape[0]):
    for j in range(a_grid.shape[1]):
        loss_grid[i, j] = loss(y, model(x, a_grid[i, j], b_grid[i, j]))

plt.figure(figsize = (8, 5))
plt.contourf(a_grid, b_grid, np.log(loss_grid), levels = 30, cmap = 'viridis')
plt.colorbar(label = 'log of the loss')
plt.plot(path_a, path_b, color = 'red', marker = '.', label = 'Gradient descent')
plt.xlabel('slope a')
plt.ylabel('intercept b')
plt.title('Walking downhill on the loss')
plt.legend()
plt.show()

In [ ]:
# Final line
y_pred = model(x, a, b)
print_table(x, y, y_pred)
plot_loss(x, y, y_pred, f'After gradient descent: y_pred = {a:.2f} x + {b:.2f}')

In [ ]:
# Check with the best fit formula of chapter 1 and with numpy implementation
x_mean = np.mean(x)
y_mean = np.mean(y)
a_formula = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
b_formula = y_mean - a_formula * x_mean
a_np, b_np = np.polyfit(x, y, deg = 1)

print(f"Gradient descent:  y_pred = {a:.4f} x + {b:.4f}")
print(f"Best fit formula:  y_pred = {a_formula:.4f} x + {b_formula:.4f}")
print(f"numpy:             y_pred = {a_np:.4f} x + {b_np:.4f}")
print("Close: ", np.isclose(a, a_formula) and np.isclose(b, b_formula))